In [14]:
import os
import time
import torch
import torch.nn.functional as F
import torchaudio
from torchaudio import transforms
import numpy as np
import bc_resnet_model
from IPython.display import Audio

In [24]:
# ================================
# 설정
# ================================
EPS         = 1e-9
SAMPLE_RATE = 16000
TARGET_LEN  = 16000  # 1초
SCALE       = 2
N_CLASS     = 2
CLASSES     = ['non-wakeword', 'wakeword']
THRESHOLD   = 0.8

# ================================
# 모델 로드
# ================================
device = torch.device('cpu')
model  = bc_resnet_model.BcResNetModel(
    n_class=N_CLASS,
    scale=SCALE,
    use_subspectral=True
)

import torch.nn as nn
model.head_conv = nn.Conv2d(32 * SCALE, N_CLASS, kernel_size=1)
model.load_state_dict(torch.load('./best_horiya2_3.pt', map_location='cpu'))
model.eval()
print("모델 로드 완료!")

# ================================
# 전처리 (re9ulus 방식)
# ================================
to_mel = transforms.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_fft=1024,
    f_max=8000,
    n_mels=40
)

def preprocess(waveform, sr):
    # 스테레오 → 모노
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    # 리샘플링
    if sr != SAMPLE_RATE:
        waveform = transforms.Resample(sr, SAMPLE_RATE)(waveform)

    # 패딩/자르기 (1초)
    length = waveform.shape[1]
    if length > TARGET_LEN:
        waveform = waveform[:, :TARGET_LEN]
    elif length < TARGET_LEN:
        waveform = F.pad(waveform, (0, TARGET_LEN - length))

    # 로그 멜 스펙트로그램
    log_mel = (to_mel(waveform) + EPS).log2()
    return log_mel.unsqueeze(0)  # (1, 1, 40, T)

def preprocess2(waveform, sr):
    # 스테레오 → 모노
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    # 리샘플링
    if sr != SAMPLE_RATE:
        waveform = transforms.Resample(sr, SAMPLE_RATE)(waveform)

    # 패딩/자르기 (1초)
    length = waveform.shape[1]
    if length > TARGET_LEN:
        waveform = waveform[:, :TARGET_LEN]
    elif length < TARGET_LEN:
        waveform = F.pad(waveform, (0, TARGET_LEN - length))

    return waveform

모델 로드 완료!


In [ ]:
base_path = "/home/isol/work/bc_resnet_re9ulus/test_audio"
wav_path_list = [os.path.join(base_path, i) for i in os.listdir(base_path)]

waveform, sr = torchaudio.load(wav_path_list[0])
Audio(preprocess(waveform, sr=sr), rate=sr)

In [42]:
base_path = "/home/isol/work/bc_resnet_re9ulus/test_audio"
wav_path_list = [os.path.join(base_path, i) for i in os.listdir(base_path)]

waveform, sr = torchaudio.load(wav_path_list[0])
new_waveform = preprocess2(waveform, sr=sr)
print(len(new_waveform[0])/16000)
Audio(new_waveform, rate=sr)

1.0


In [13]:
# ================================
# 테스트
# ================================

base_path = "/home/isol/work/bc_resnet_re9ulus/test_audio"
wav_path_list = [os.path.join(base_path, i) for i in os.listdir(base_path)]

inference_ms_list = []

for wav_path in wav_path_list:
    waveform, sr = torchaudio.load(wav_path)
    spec = preprocess(waveform, sr)

    with torch.no_grad():
        start = time.time()
        output = model(spec)
        end = time.time()
        probs  = F.softmax(output.squeeze(), dim=0)

    inference_ms = (end - start) * 1000
    inference_ms_list.append(inference_ms)

    print("="*35)
    print(f"파일 이름: {wav_path}")
    for i, cls in enumerate(CLASSES):
        bar    = "█" * int(probs[i].item() * 20)
        marker = " ← 감지!" if (cls == 'wakeword' and probs[i].item() >= THRESHOLD) else ""
        print(f"{cls:10s}: {probs[i].item()*100:5.1f}% {bar}{marker}")
    print("="*35)

mean_inference_ms = np.mean(inference_ms_list)
print(f"평균 추론 시간: {mean_inference_ms:.2f}ms")
print(f"30ms 실시간 가능 여부: {'✅ 가능' if mean_inference_ms < 30 else '❌ 느림, 스텝 늘려야 함'}")

파일 이름: /home/isol/work/bc_resnet_re9ulus/test_audio/raw_ring_20260304_165658_000000.wav
non-wakeword:  68.7% █████████████
wakeword  :  31.3% ██████
파일 이름: /home/isol/work/bc_resnet_re9ulus/test_audio/raw_ring_20260304_165952_000001.wav
non-wakeword:  77.2% ███████████████
wakeword  :  22.8% ████
파일 이름: /home/isol/work/bc_resnet_re9ulus/test_audio/raw_ring_20260304_165252_000000.wav
non-wakeword:   0.5% 
wakeword  :  99.5% ███████████████████ ← 감지!
파일 이름: /home/isol/work/bc_resnet_re9ulus/test_audio/raw_ring_20260304_164109_000004.wav
non-wakeword:   2.1% 
wakeword  :  97.9% ███████████████████ ← 감지!
파일 이름: /home/isol/work/bc_resnet_re9ulus/test_audio/raw_ring_20260304_170000_000002.wav
non-wakeword:   1.5% 
wakeword  :  98.5% ███████████████████ ← 감지!
파일 이름: /home/isol/work/bc_resnet_re9ulus/test_audio/raw_ring_20260304_165918_000000.wav
non-wakeword:   0.9% 
wakeword  :  99.1% ███████████████████ ← 감지!
평균 추론 시간: 2.20ms
30ms 실시간 가능 여부: ✅ 가능
